# ASR éwé v0.1 — Notebook 07 : fine-tuning whisper-small sur WAXAL

Projet : **tg_nlp_toolkit** — brique ASR. Contexte, décisions et licences :
`docs/14-asr-ewe-plan-execution-v0.1.md`. À lancer **après** le notebook 06
(ses baselines sont lues à la fin pour la comparaison).

**Décisions v0.1** :

- base `openai/whisper-small` (244 M, licence MIT) — comparable à la référence publique WAXALNet ;
- entraînement complet (pas de LoRA) : batch effectif 16, lr 1e-4, warmup 500 pas (point de départ : model card WAXALNet), 3 époques avec arrêt précoce sur le WER de validation ;
- clips filtrés : transcription non vide, durée entre 1,5 s et 30 s (limite d'entrée Whisper) ;
- évaluation finale WER + CER sur le test complet, normalisation documentée (doc 14, section 7).

**Prérequis** : Colab T4. **Durée estimée** : ~3-4 h (préparation des données, entraînement, évaluation finale). En cas de coupure, relancer avec `trainer.train(resume_from_checkpoint=True)`.




In [ ]:
!pip install -q -U datasets soundfile scipy transformers accelerate jiwer tensorboard




In [ ]:
import io
import json
import re
import time
import unicodedata
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Union

import jiwer
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from scipy import signal

warnings.filterwarnings("ignore", message=".*Xing.*")

REPO = "google/WaxalNLP"
LANGUE = "ewe"
TACHE = "ASR"
MODELE_BASE = "openai/whisper-small"
SEED = 42

# Hyperparametres (voir doc 14, section 5).
TAILLE_LOT = 8           # par appareil
ACCUM_GRAD = 2           # lot effectif = 16
TAUX_APPRENTISSAGE = 1e-4
EPOCHS = 3
PAS_ECHAUFFEMENT = 500
PAS_EVAL = 500           # evaluation + checkpoint tous les 500 pas
TAILLE_EVAL = 250        # sous-echillon du validation pour l'arret precoce
DUREE_MIN, DUREE_MAX = 1.5, 30.0

SORTIES = Path("resultats")
SORTIES.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Materiel :", DEVICE, "| torch", torch.__version__)

torch.manual_seed(SEED)
np.random.seed(SEED)




In [ ]:
# Fonctions communes (egalement dans src/asr/prep_waxal_ewe.py,
# version canonique : manifeste + filtres documentes).

MOTIF_FICHIER = __import__("re").compile(r"^(?P<langue>[a-z]{3})-(?P<split>[a-z]+)-\d+\.parquet$")


def fichiers_parquet(split):
    """Liste triee des parquet d'un split, via l'API Hugging Face."""
    prefixe = f"data/{TACHE}/{LANGUE}/"
    noms = [
        f for f in HfApi().list_repo_files(REPO, repo_type="dataset")
        if f.startswith(prefixe) and f.endswith(".parquet")
        and MOTIF_FICHIER.match(f.split("/")[-1])
    ]
    attends = [f for f in noms if f.split("/")[-1].startswith(f"{LANGUE}-{split}-")]
    if not attends:
        raise FileNotFoundError(
            f"Aucun parquet pour {REPO} / {LANGUE}_{TACHE} / {split}"
        )
    return sorted(attends)


def charger_split(split):
    """Charge un split via le builder parquet generique.

    La config `ewe_asr` declaree dans le YAML du dataset echoue au
    chargement (colonne `__index_level_0__` absente du schema declare) ;
    on lit donc les fichiers parquet directement.
    """
    urls = [f"hf://datasets/{REPO}@main/{f}" for f in fichiers_parquet(split)]
    return load_dataset("parquet", data_files=urls, split="train")


def duree_secondes(extrait):
    """Duree en secondes, lue dans les octets audio bruts (MP3)."""
    audio = extrait.get("audio")
    if not isinstance(audio, dict) or not audio.get("bytes"):
        return None
    try:
        infos = sf.info(io.BytesIO(audio["bytes"]))
    except (sf.LibsndfileError, RuntimeError):
        return None
    if not infos.samplerate:
        return None
    return infos.frames / infos.samplerate


def charger_audio_16k(extrait):
    """Decode le MP3, moyenne les canaux, reechantillonne a 16 kHz.

    Renvoie un tableau float32 1-D a 16 kHz, ou None si l'audio est
    illisible. Le format source est MP3 44,1 kHz stereo (verifie le
    2026-11-16).
    """
    audio = extrait.get("audio")
    if not isinstance(audio, dict) or not audio.get("bytes"):
        return None
    try:
        piste, frequence = sf.read(
            io.BytesIO(audio["bytes"]), dtype="float32", always_2d=True
        )
    except (sf.LibsndfileError, RuntimeError):
        return None
    mono = piste.mean(axis=1)
    if frequence != 16000:
        mono = signal.resample_poly(mono, 16000, frequence)
    return mono.astype(np.float32)


def normaliser(texte):
    """Normalisation commune a toutes les metriques (documentee doc 14,
    section 7) : NFKC, minuscules, ponctuation retiree, espaces replies."""
    texte = unicodedata.normalize("NFKC", str(texte)).lower()
    texte = re.sub(r"[^\w\s]", " ", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte.strip()


def wer_cer(references, predictions):
    """WER et CER agreges sur le corpus (jiwer), apres normalisation."""
    paires = [(normaliser(r), normaliser(p)) for r, p in zip(references, predictions)]
    paires = [(r, p) for r, p in paires if r]
    refs, hyps = zip(*paires) if paires else ([], [])
    if not refs:
        return 1.0, 1.0
    return jiwer.wer(list(refs), list(hyps)), jiwer.cer(list(refs), list(hyps))


def wer_par_extrait(references, predictions):
    """WER par extrait (pour l'analyse des pires erreurs)."""
    scores = []
    for r, p in zip(references, predictions):
        rn = normaliser(r)
        if not rn:
            continue
        pn = normaliser(p) or " "
        scores.append(jiwer.wer(rn, pn))
    return scores




In [ ]:
# Chargement et filtrage (meme logique que src/asr/prep_waxal_ewe.py).
brut = {s: charger_split(s) for s in SPLITS}


def filtrer(dataset):
    indices = []
    motifs = {"transcription_vide": 0, "audio_illisible": 0,
              "duree_courte": 0, "duree_longue": 0}
    for i, ex in enumerate(dataset):
        texte = str(ex.get("transcription") or "").strip()
        d = duree_secondes(ex)
        if not texte:
            motifs["transcription_vide"] += 1
            continue
        if d is None:
            motifs["audio_illisible"] += 1
            continue
        if d < DUREE_MIN:
            motifs["duree_courte"] += 1
            continue
        if d > DUREE_MAX:
            motifs["duree_longue"] += 1
            continue
        indices.append(i)
    return dataset.select(indices), motifs

avant = {s: len(brut[s]) for s in SPLITS}
donnees = {}
for s in SPLITS:
    donnees[s], motifs = filtrer(brut[s])
    print(f"{s:12s} {avant[s]:6d} -> {len(donnees[s]):6d} extraits | motifs : {motifs}")




In [ ]:
# Preparation : log-mel + tokenisation. Decode le MP3 (mono 16 kHz) via
# charger_audio_16k. Le cache Arrow sur disque grossit (~15 Go pour le train).
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODELE_BASE)
# Note : le tokenizer d'openai/whisper-small definit deja un pad token
# (verifie le 2026-11-16) ; aucun reglage n'est necessaire.


def preparer(exemples):
    audios = [charger_audio_16k(a) for a in exemples["audio"]]
    entrees = processor(
        audios, sampling_rate=16000, return_tensors="np"
    ).input_features
    etiquettes = processor.tokenizer(exemples["transcription"]).input_ids
    return {"input_features": entrees, "labels": etiquettes}


t0 = time.time()
for s in SPLITS:
    donnees[s] = (
        donnees[s]
        .remove_columns([c for c in donnees[s].column_names
                         if c not in ("transcription", "audio")])
        .map(preparer, batched=True, batch_size=8,
             remove_columns=["transcription", "audio"])
    )
    print(f"{s} prepare en {time.time() - t0:.0f} s")




In [ ]:
# Collecteur de lot : padding des log-mel (fixe a 30 s) et des etiquettes.
@dataclass
class CollecteurBatch:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        entrees = [{"input_features": f["input_features"]} for f in features]
        lot = self.processor.feature_extractor.pad(entrees, return_tensors="pt")
        etiquettes = [{"input_ids": f["labels"]} for f in features]
        lot_etiquettes = self.processor.tokenizer.pad(
            etiquettes, padding=True, return_tensors="pt"
        )
        lot["labels"] = lot_etiquettes["input_ids"].masked_fill(
            lot_etiquettes["attention_mask"].eq(0), -100
        )
        return lot




In [ ]:
# Metriques : WER + CER sur texte normalise (doc 14, section 7).
def calculer_metriques(p):
    ids_predits = p.predictions
    preds = processor.tokenizer.batch_decode(ids_predits, skip_special_tokens=True)

    etiquettes = np.where(
        p.label_ids != -100, p.label_ids, processor.tokenizer.pad_token_id
    )
    refs = processor.tokenizer.batch_decode(etiquettes, skip_special_tokens=True)

    paires = [(normaliser(r), normaliser(h)) for r, h in zip(refs, preds)]
    paires = [(r, h) for r, h in paires if r]
    if not paires:
        return {"wer": 1.0, "cer": 1.0}
    refs_n, preds_n = zip(*paires)
    return {
        "wer": jiwer.wer(list(refs_n), list(preds_n)),
        "cer": jiwer.cer(list(refs_n), list(preds_n)),
    }




In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
)

modele = WhisperForConditionalGeneration.from_pretrained(MODELE_BASE)
modele.config.use_cache = False  # requis avec gradient checkpointing

eval_validation = donnees["validation"].shuffle(seed=SEED).select(
    range(min(TAILLE_EVAL, len(donnees["validation"])))
)

args = Seq2SeqTrainingArguments(
    output_dir="checkpoints-whisper-small-ewe",
    per_device_train_batch_size=TAILLE_LOT,
    gradient_accumulation_steps=ACCUM_GRAD,
    learning_rate=TAUX_APPRENTISSAGE,
    warmup_steps=PAS_ECHAUFFEMENT,
    num_train_epochs=EPOCHS,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_strategy="steps",      # anciennes versions : evaluation_strategy
    eval_steps=PAS_EVAL,
    save_strategy="steps",
    save_steps=PAS_EVAL,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=50,
    report_to="tensorboard",
    seed=SEED,
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=modele,
    args=args,
    train_dataset=donnees["train"],
    eval_dataset=eval_validation,
    data_collator=CollecteurBatch(processor),
    compute_metrics=calculer_metriques,
    processing_class=processor,   # anciennes versions : tokenizer=processor
)
print("Pret. Pas par epoque :", -(-len(donnees["train"]) // (TAILLE_LOT * ACCUM_GRAD)))




In [ ]:
# Entrainement (~2 h pour 3 epoques sur T4).
# Reprise apres coupure : trainer.train(resume_from_checkpoint=True)
t0 = time.time()
trainer.train()
print(f"Entrainement termine en {(time.time() - t0) / 3600:.2f} h")




In [ ]:
# Evaluation finale sur le test complet (WER + CER).
resultat_test = trainer.predict(donnees["test"])
print("Metriques du test :", resultat_test.metrics)

ids_predits = resultat_test.predictions
preds = processor.tokenizer.batch_decode(ids_predits, skip_special_tokens=True)

etiquettes = np.where(
    resultat_test.label_ids != -100,
    resultat_test.label_ids,
    processor.tokenizer.pad_token_id,
)
refs = processor.tokenizer.batch_decode(etiquettes, skip_special_tokens=True)

wer_final, cer_final = wer_cer(refs, preds)
print(f"Notre modele v0.1 : WER {wer_final:.1%} | CER {cer_final:.1%} "
      f"({len(refs)} extraits)")

# Pires erreurs : a faire relire par un locuteur (analyse qualitative).
df_test = pd.DataFrame({"reference": refs, "prediction": preds})
df_test["wer_extrait"] = wer_par_extrait(refs, preds)
df_test.sort_values("wer_extrait", ascending=False).head(20).to_csv(
    SORTIES / "pires_erreurs_test.csv", index=False
)
print("20 pires erreurs exportees dans resultats/pires_erreurs_test.csv")




In [ ]:
# Comparaison avec les baselines du notebook 06 (si le fichier est present).
chemin_baselines = SORTIES / "baselines_test.json"
lignes = []
if chemin_baselines.exists():
    lignes = json.loads(chemin_baselines.read_text(encoding="utf-8"))
lignes.append({
    "modele": "whisper-small fine-tune WAXAL ewe (notre v0.1)",
    "wer": wer_final,
    "cer": cer_final,
    "n": len(refs),
})
tableau = pd.DataFrame(lignes)
tableau.to_json(SORTIES / "comparaison_test.json", orient="records", indent=2)
tableau




In [ ]:
# Sauvegarde locale du modele et du processor.
REPERTOIRE_MODELE = "asr-ewe-v0.1-whisper-small"
modele.save_pretrained(REPERTOIRE_MODELE)
processor.save_pretrained(REPERTOIRE_MODELE)
print("Modele sauvegarde dans", REPERTOIRE_MODELE)




## Publication sur Hugging Face (optionnelle)

Le modele peut etre publie sous licence libre avec attribution (base MIT +
donnees WAXAL ; voir doc 14, section 6 pour la nuance BY vs BY-SA).

```python
from huggingface_hub import login
login()  # jeton en ecriture
modele.push_to_hub("cheriftenga/whisper-small-ewe-v0.1")
processor.push_to_hub("cheriftenga/whisper-small-ewe-v0.1")
```

Model card minimale a coller :

- **Donnees d'entrainement** : WAXAL `ewe_asr` (Google Research et partenaires, CC-BY-4.0), parole spontanee image-prompted, clips filtres 1,5-30 s.
- **Base** : openai/whisper-small (MIT).
- **Evaluation** : WER / CER sur le test public complet (1 891 extraits), normalisation documentee dans le depot.
- **Limites connues** : locuteurs probablement ghanéens (variante du Ghana), parole spontanee bruyante, pas de double validation des transcriptions ; ne pas utiliser en production sans validation locale.
- **Licence** : CC-BY-4.0 (ou CC-BY-SA-4.0 selon l'audit licence WAXAL, doc 14, section 6).




## Après l'exécution — checklist

1. Noter WER/CER final + baselines dans `docs/17-asr-ewe-resultats-v0.1.md` (avec date, hyperparamètres, durée d'entraînement).
2. Télécharger depuis la VM : `resultats/` (comparaison, pires erreurs), le dossier `asr-ewe-v0.1-whisper-small/`, et le meilleur checkpoint si besoin.
3. Faire relire les 20 pires erreurs à un locuteur éwé (analyse qualitative, comme la double validation du projet).
4. Publier le modèle sur Hugging Face (cellule ci-dessus) et mettre à jour le README du toolkit.
5. Décider des prochaines étapes v0.2 : test togolais, pseudo-labeling des 183 920 extraits non transcrits, essai MMS-adapters.


